# Posterior Predictive Checks: modelos base

**Etapa del workflow bayesiano: Model Checking (Gelman et al. 2020)**

La inferencia bayesiana produce una distribución sobre los parámetros, pero no garantiza que el modelo sea adecuado. Los **Posterior Predictive Checks (PPCs)** responden la pregunta de validación: ¿puede el modelo generar datos que se parezcan a los observados?

La idea es concreta: para cada muestra posterior $(\alpha^{(s)}, \beta^{(s)})$, simulamos un dataset replicado $y^{\text{rep}(s)}$ desde la distribución del modelo. Si el modelo es una buena descripción de los datos, la distribución de $y^{\text{rep}}$ debe envolver a $y^{\text{obs}}$ en las dimensiones que nos importan.

Para datos de conteo, las dimensiones críticas son:
- **Dispersión:** ¿replica el modelo la variabilidad observada? Poisson está estructuralmente limitado a $\text{Var}[Y] = \mathbb{E}[Y]$.
- **Colas:** ¿genera el modelo valores extremos similares a los observados?
- **Proporción de ceros:** ¿predice el modelo la abundancia de hembras sin satélites?

> **Prerequisito:** ejecutar `03_bayesian_inference` para generar `outputs/pois_idata.nc` y `outputs/neg_idata.nc`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import arviz as az

sns.set_style('whitegrid')
np.random.seed(42)

In [ ]:
# Cargar InferenceData con posterior predictive
idata_pois = az.from_netcdf('../outputs/pois_idata.nc')
idata_nb   = az.from_netcdf('../outputs/neg_idata.nc')

# az.plot_ppc exige que la var en posterior_predictive coincida con observed_data ('y')
idata_pois.posterior_predictive = idata_pois.posterior_predictive.rename({'y_rep': 'y'})
idata_nb.posterior_predictive   = idata_nb.posterior_predictive.rename({'y_rep': 'y'})

# Datos observados
df = pd.read_csv('../data/agresti_crab_satellites_dataset.csv')
y_obs = df['Sa'].values

print(f"Grupos disponibles Poisson:  {list(idata_pois.groups())}")
print(f"Grupos disponibles NegBin:   {list(idata_nb.groups())}")

## 1. Verificación de sobredispersión

Antes de los PPCs, confirmamos cuantitativamente la sobredispersión en los datos. Bajo Poisson, la varianza condicional es igual a la media: $\text{Var}[Y \mid X] = \mathbb{E}[Y \mid X]$, lo que implica un índice de dispersión $D = \text{Var}/\text{Mean} \approx 1$.

Si $D \gg 1$, el modelo Poisson subestimará la variabilidad y producirá predicciones demasiado concentradas — diagnóstico que ya observamos en `01_basic_eda`.

In [ ]:
mean_obs = np.mean(y_obs)
var_obs  = np.var(y_obs)
D_obs    = var_obs / mean_obs  # Índice de dispersión (VMR)

print(f"Media observada:              {mean_obs:.3f}")
print(f"Varianza observada:           {var_obs:.3f}")
print(f"Índice de dispersión (D=V/M): {D_obs:.3f}  {'← sobredispersión' if D_obs > 1 else '← equidispersión'}")
print(f"Proporción de ceros:          {np.mean(y_obs == 0):.3f}")
print(f"Valor máximo:                 {np.max(y_obs)}")

## 2. PPC visual — distribución marginal

La función `az.plot_ppc` superpone la distribución del dato observado $y^{\text{obs}}$ (línea gruesa) sobre la nube de distribuciones replicadas $y^{\text{rep}}$ (líneas delgadas, una por muestra). Si el modelo es adecuado, $y^{\text{obs}}$ debe quedar envuelta por la nube — no en la cola ni fuera de ella.

Con datos de conteo, el gráfico de densidad continua de ArviZ puede ocultar detalles en los valores enteros bajos (especialmente en los ceros). Complementamos con los estadísticos de prueba de la sección siguiente.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

az.plot_ppc(idata_pois, ax=axes[0], num_pp_samples=100)
axes[0].set_title('Poisson')
axes[0].set_xlabel('Número de satélites')

az.plot_ppc(idata_nb, ax=axes[1], num_pp_samples=100)
axes[1].set_title('Binomial Negativa')
axes[1].set_xlabel('Número de satélites')
axes[1].set_xlim(0, 17.5)

plt.suptitle('PPC: Distribución marginal de y_rep vs y_obs', fontsize=13)
plt.tight_layout()
plt.show()

## 3. PPCs con estadísticos de prueba

Para cada estadístico de prueba $T$, comparamos $T(y^{\text{obs}})$ con la distribución de $T(y^{\text{rep}})$ sobre las 8000 muestras posteriores. El **Bayesian p-value** $p_B = P(T(y^{\text{rep}}) \geq T(y^{\text{obs}}))$ cuantifica si la observación es atípica bajo el modelo: valores cerca de 0 o 1 indican que el modelo no captura esa propiedad en esa dirección.

Usamos cuatro estadísticos orientados a los problemas conocidos de estos modelos:
- `std`: dispersión total — el fallo más directo de Poisson.
- `Índice de dispersión` ($D = \text{Var}/\text{Mean}$): directamente el supuesto que Poisson viola.
- `max`: sensible a colas pesadas — ¿genera el modelo valores extremos?
- `prop. ceros`: detecta inflación de ceros por encima de lo que predice la distribución base.

In [ ]:
def dispersion_index(y):
    """Índice de dispersión: Var/Mean. Bajo Poisson debe ser ~1."""
    return np.var(y) / np.mean(y)

def prop_zeros(y):
    """Proporción de ceros."""
    return np.mean(y == 0)

test_stats = {
    'std':               np.std,
    'Índice dispersión': dispersion_index,
    'max':               np.max,
    'prop. ceros':       prop_zeros,
}

In [ ]:
# Extraer muestras y_rep
y_rep_pois = idata_pois.posterior_predictive['y'].values  # shape: (chains, draws, N)
y_rep_nb   = idata_nb.posterior_predictive['y'].values

# Aplanar chains y draws
y_rep_pois_flat = y_rep_pois.reshape(-1, y_rep_pois.shape[-1])  # (S, N)
y_rep_nb_flat   = y_rep_nb.reshape(-1, y_rep_nb.shape[-1])

print(f"y_rep_pois shape: {y_rep_pois_flat.shape}")
print(f"y_rep_nb shape:   {y_rep_nb_flat.shape}")

In [ ]:
fig, axes = plt.subplots(len(test_stats), 2, figsize=(12, 4 * len(test_stats)))

for row, (stat_name, stat_fn) in enumerate(test_stats.items()):
    t_obs = stat_fn(y_obs)

    for col, (y_rep_flat, model_name) in enumerate([
        (y_rep_pois_flat, 'Poisson'),
        (y_rep_nb_flat,   'Binomial Negativa')
    ]):
        t_rep = np.array([stat_fn(y_rep_flat[s]) for s in range(y_rep_flat.shape[0])])
        p_val = np.mean(t_rep >= t_obs)

        ax = axes[row, col]
        ax.hist(t_rep, bins=40, color='steelblue', alpha=0.7, label='T(y_rep)')
        ax.axvline(t_obs, color='tomato', lw=2, label=f'T(y_obs) = {t_obs:.2f}')
        ax.set_title(f'{model_name} — {stat_name}\nBayesian p-value = {p_val:.3f}')
        ax.legend(fontsize=8)

plt.suptitle('PPCs con estadísticos de prueba', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 4. LOO-PIT — calibración predictiva

El LOO-PIT (*Leave-One-Out Probability Integral Transform*) evalúa calibración: para cada observación $y_i$, calcula el cuantil de $y_i$ dentro de su distribución predictiva leave-one-out. Si el modelo está bien calibrado, esos cuantiles deben ser aproximadamente uniformes — el histograma debe ser plano y la curva de diferencia debe mantenerse cerca de cero.

Una desviación sistemática hacia la izquierda indica que el modelo sobreestima los valores (predice demasiados ceros relativos); hacia la derecha, subestima la masa en valores bajos.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

az.plot_loo_pit(idata_pois, y='y', ax=axes[0])
axes[0].set_title('Poisson — LOO-PIT')

az.plot_loo_pit(idata_nb, y='y', ax=axes[1])
axes[1].set_title('Binomial Negativa — LOO-PIT')

plt.tight_layout()
plt.show()

## 5. Comparación de modelos — LOO-CV

La validación cruzada leave-one-out estima la capacidad predictiva out-of-sample de cada modelo. ArviZ calcula el **ELPD** (*Expected Log Predictive Density*): valores más altos (menos negativos) indican mejor predicción. La diferencia ELPD ± SE permite saber si la diferencia es estadísticamente sustancial o solo ruido de estimación.

In [ ]:
comparison = az.compare({'Poisson': idata_pois, 'NegBin': idata_nb})
display(comparison)

az.plot_compare(comparison, figsize=(8, 3))
plt.title('Comparación LOO-CV: Poisson vs Binomial Negativa')
plt.tight_layout()
plt.show()

## 6. Resumen de evidencia

Consolidamos los Bayesian p-values de los cuatro estadísticos de prueba para ambos modelos. Heurística de interpretación: $p_B < 0.05$ o $p_B > 0.95$ indican fallo claro del modelo en esa dimensión.

In [ ]:
# Tabla resumen
rows = []
for stat_name, stat_fn in test_stats.items():
    t_obs = stat_fn(y_obs)
    for y_rep_flat, model_name in [
        (y_rep_pois_flat, 'Poisson'),
        (y_rep_nb_flat,   'NegBin')
    ]:
        t_rep = np.array([stat_fn(y_rep_flat[s]) for s in range(y_rep_flat.shape[0])])
        rows.append({
            'Modelo': model_name,
            'Estadístico': stat_name,
            'T(y_obs)': round(t_obs, 3),
            'E[T(y_rep)]': round(np.mean(t_rep), 3),
            'Bayesian p-value': round(np.mean(t_rep >= t_obs), 3),
        })

summary_df = pd.DataFrame(rows).sort_values(['Estadístico', 'Modelo'])
display(summary_df)

### Conclusiones

#### Modelo Poisson — falla sistemática

El modelo Poisson falla en todas las dimensiones evaluadas:

- **Sobredispersión** ($p_B \approx 0.000$ para std e índice de dispersión): el modelo predice una dispersión de ~2.0 satélites cuando los datos muestran ~3.1. El índice de dispersión observado es $D = 3.38$, pero Poisson lo fija estructuralmente en $D \approx 1$.
- **Inflación de ceros** ($p_B \approx 0.000$): el 35.8% de las hembras no tiene satélites, pero Poisson predice apenas ~8.2%. El modelo subestima masivamente los ceros.
- **Colas pesadas** ($p_B = 0.068$): marginalmente, también le cuesta llegar al valor máximo observado (15 satélites).
- **LOO-CV**: ELPD = −465.4, muy por debajo de la Binomial Negativa.

#### Modelo Binomial Negativa — mejora en dispersión, falla residual en ceros

La Binomial Negativa resuelve parcialmente el problema:

- **Sobredispersión capturada**: $p_B = 0.894$ para std y $p_B = 0.954$ para índice de dispersión — el modelo envuelve correctamente la variabilidad de los datos.
- **Colas pesadas**: $p_B = 0.968$, aunque $\mathbb{E}[\max(y^{\text{rep}})] \approx 27$ sobreestima el máximo observado (15), lo que sugiere captura de colas con exceso.
- **Inflación de ceros — falla persistente** ($p_B = 0.093$): aunque mejora respecto a Poisson ($\mathbb{E}[\text{prop\_zeros}] \approx 29.1\%$ vs $8.2\%$), todavía subestima la proporción observada de 35.8%.
- **LOO-CV**: ELPD = −378.5 vs −465.4 de Poisson. La diferencia de 86.8 puntos (±17.3 SE) es sustancial y estadísticamente clara.

#### Diagnóstico final y extensión natural

Ambos modelos comparten el mismo predictor lineal y ninguno tiene un mecanismo explícito para el exceso de ceros. Hay hembras que estructuralmente no atraen satélites — posiblemente por condición física o tamaño muy reducido — y ese proceso generador de ceros es distinto del proceso de conteo.

La extensión natural es un **modelo de mezcla con inflación de ceros**:

- **Zero-Inflated Poisson (ZIP)**: mezcla de masa puntual en cero (probabilidad $\pi$) y Poisson($\lambda$). Aborda la inflación de ceros pero no la sobredispersión residual.
- **Zero-Inflated Negative Binomial (ZINB)**: mezcla de masa en cero y NegBin($\mu$, $\phi$). Aborda simultáneamente la inflación de ceros y la sobredispersión — es el candidato más natural dado el diagnóstico.

En ambos casos el modelo tiene dos componentes: uno que decide si la hembra "puede" tener satélites (componente logístico sobre $W$), y otro que modela cuántos dado que puede (Poisson o NegBin sobre $W$).

---

## ¿Qué sigue?

El notebook `05_predictive_checks_extended_models` implementa ZIP y ZINB con Stan, aplica los mismos PPCs para verificar si resuelven el problema de inflación de ceros, y compara los cuatro modelos con LOO-CV.